# 🔐 Phishing URL Detector
This notebook extracts features from a URL and uses a trained XGBoost model to classify it as **Safe** or **Phishing**.

**Steps:**
1. Install dependencies
2. Upload your model file (`XGBoostClassifier.pickle.dat`)
3. Define feature extraction functions
4. Enter a URL and get the prediction

In [1]:
# ============================================================
# CELL 1: Install required packages
# ============================================================
!pip install python-whois -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 117.0/117.0 kB 4.0 MB/s eta 0:00:00


In [2]:
# ============================================================
# CELL 2: Upload your XGBoost model file
# Run this cell, then upload 'XGBoostClassifier.pickle.dat'
# ============================================================
from google.colab import files
uploaded = files.upload()  # Upload XGBoostClassifier.pickle.dat here

Saving XGBoostClassifier.pickle.dat to XGBoostClassifier.pickle.dat


In [3]:
# ============================================================
# CELL 3: Import libraries & define all feature functions
# ============================================================
import re
import socket
import requests
import whois
import numpy as np
import ipaddress
from urllib.parse import urlparse
from datetime import datetime

# ----------------------------------------------------------
# 1. Domain of the URL
# ----------------------------------------------------------
def getDomain(url):
    domain = urlparse(url).netloc
    if re.match(r"^www\.", domain):
        domain = re.sub(r"^www\.", "", domain)
    return domain

# ----------------------------------------------------------
# 2. IP Address in the URL (Have_IP)
# Returns 1 if IP used instead of domain (phishing indicator)
# ----------------------------------------------------------
def havingIP(url):
    try:
        host = urlparse(url).netloc  # extract just the host
        ipaddress.ip_address(host)
        return 1
    except:
        return 0

# ----------------------------------------------------------
# 3. '@' Symbol in URL (Have_At)
# Browser ignores everything before '@' — common in phishing
# ----------------------------------------------------------
def haveAtSign(url):
    return 1 if "@" in url else 0

# ----------------------------------------------------------
# 4. Length of URL (URL_Length)
# URL >= 54 characters → phishing (1), else legitimate (0)
# ----------------------------------------------------------
def getLength(url):
    return 0 if len(url) < 54 else 1

# ----------------------------------------------------------
# 5. Depth of URL (URL_Depth)
# Counts sub-pages based on '/' separators
# ----------------------------------------------------------
def getDepth(url):
    s = urlparse(url).path.split('/')
    return sum(1 for j in s if len(j) != 0)

# ----------------------------------------------------------
# 6. Redirection '//' in URL (Redirection)
# '//' beyond position 7 indicates redirection
# ----------------------------------------------------------
def redirection(url):
    pos = url.rfind('//')
    return 1 if pos > 7 else 0

# ----------------------------------------------------------
# 7. 'http/https' in Domain Name (https_Domain)
# Phishers add HTTPS token to domain to trick users
# ----------------------------------------------------------
def httpDomain(url):
    return 1 if 'https' in urlparse(url).netloc else 0

# ----------------------------------------------------------
# 8. URL Shortening Services (TinyURL)
# ----------------------------------------------------------
shortening_services = (
    r"bit\.ly|goo\.gl|shorte\.st|go2l\.ink|x\.co|ow\.ly|t\.co|tinyurl|tr\.im|is\.gd|cli\.gs|"
    r"yfrog\.com|migre\.me|ff\.im|tiny\.cc|url4\.eu|twit\.ac|su\.pr|twurl\.nl|snipurl\.com|"
    r"short\.to|BudURL\.com|ping\.fm|post\.ly|Just\.as|bkite\.com|snipr\.com|fic\.kr|loopt\.us|"
    r"doiop\.com|short\.ie|kl\.am|wp\.me|rubyurl\.com|om\.ly|to\.ly|bit\.do|t\.co|lnkd\.in|db\.tt|"
    r"qr\.ae|adf\.ly|goo\.gl|bitly\.com|cur\.lv|tinyurl\.com|ow\.ly|bit\.ly|ity\.im|q\.gs|is\.gd|"
    r"po\.st|bc\.vc|twitthis\.com|u\.to|j\.mp|buzurl\.com|cutt\.us|u\.bb|yourls\.org|x\.co|"
    r"prettylinkpro\.com|scrnch\.me|filoops\.info|vzturl\.com|qr\.net|1url\.com|tweez\.me|v\.gd|"
    r"tr\.im|link\.zip\.net"
)

def tinyURL(url):
    return 1 if re.search(shortening_services, url) else 0

# ----------------------------------------------------------
# 9. Prefix/Suffix '-' in Domain (Prefix_Suffix)
# '-' in domain is rare in legitimate URLs
# ----------------------------------------------------------
def prefixSuffix(url):
    return 1 if '-' in urlparse(url).netloc else 0

# ----------------------------------------------------------
# 10. Web Traffic (Web_Traffic)
# Simplified: returns 1 (unknown/phishing) as default
# since Alexa API is deprecated and requires a paid key
# ----------------------------------------------------------
def web_traffic(url):
    return 1

# ----------------------------------------------------------
# 11. Age of Domain (Domain_Age)
# Age < 6 months → phishing (1), else legitimate (0)
# ----------------------------------------------------------
def domainAge(domain_info):
    try:
        creation_date = domain_info.creation_date
        expiration_date = domain_info.expiration_date
        if isinstance(creation_date, list):    creation_date = creation_date[0]
        if isinstance(expiration_date, list):  expiration_date = expiration_date[0]
        ageofdomain = abs((expiration_date - creation_date).days)
        return 1 if (ageofdomain / 30) < 6 else 0
    except:
        return 1

# ----------------------------------------------------------
# 12. End Period of Domain (Domain_End)
# Remaining time < 6 months → phishing (1), else (0)
# ----------------------------------------------------------
def domainEnd(domain):
    try:
        w = whois.whois(domain)
        expiration_date = w.expiration_date
        if isinstance(expiration_date, list): expiration_date = expiration_date[0]
        today = datetime.now()
        end = abs((expiration_date.replace(tzinfo=None) - today).days)
        return 1 if (end / 30) < 6 else 0
    except:
        return 1

# ----------------------------------------------------------
# 13. IFrame Redirection (iFrame)
# Presence of <iframe> tag → phishing (1)
# ----------------------------------------------------------
def iframe(html):
    return 1 if "<iframe" in html.lower() else 0

# ----------------------------------------------------------
# 14. Status Bar Customization via onMouseOver (Mouse_Over)
# ----------------------------------------------------------
def mouseOver(html):
    return 1 if "onmouseover" in html.lower() else 0

# ----------------------------------------------------------
# 15. Disabling Right Click (Right_Click)
# ----------------------------------------------------------
def rightClick(html):
    return 1 if "event.button==2" in html.lower() else 0

# ----------------------------------------------------------
# 16. Website Forwarding (Web_Forwards)
# ----------------------------------------------------------
def forwarding(html):
    return 1 if ("window.location" in html.lower() or "meta http-equiv" in html.lower()) else 0

print("✅ All feature functions defined successfully.")

✅ All feature functions defined successfully.


In [4]:
# ============================================================
# CELL 4: build_features — extracts all 16 features from URL
# ============================================================
def build_features(url):
    features = []
    domain = getDomain(url)

    # Fetch HTML content
    try:
        response = requests.get(url, timeout=5)
        html = response.text
    except:
        html = ""

    # WHOIS lookup
    try:
        domain_info = whois.whois(domain)
    except:
        domain_info = None

    # ── URL-based features ──────────────────────────────────
    features.append(havingIP(url))        # 1.  Have_IP
    features.append(haveAtSign(url))      # 2.  Have_At
    features.append(getLength(url))       # 3.  URL_Length
    features.append(getDepth(url))        # 4.  URL_Depth
    features.append(redirection(url))     # 5.  Redirection
    features.append(httpDomain(url))      # 6.  https_Domain
    features.append(tinyURL(url))         # 7.  TinyURL
    features.append(prefixSuffix(url))    # 8.  Prefix/Suffix

    # ── Domain-based features ───────────────────────────────
    try:                                  # 9.  DNS_Record
        socket.gethostbyname(domain)
        features.append(0)
    except:
        features.append(1)

    features.append(web_traffic(url))                           # 10. Web_Traffic
    features.append(domainAge(domain_info) if domain_info else 1)  # 11. Domain_Age
    features.append(domainEnd(domain))                          # 12. Domain_End

    # ── HTML/JS-based features ──────────────────────────────
    features.append(iframe(html))         # 13. iFrame
    features.append(mouseOver(html))      # 14. Mouse_Over
    features.append(rightClick(html))     # 15. Right_Click
    features.append(forwarding(html))     # 16. Web_Forwards

    return np.array(features).reshape(1, -1)

print("✅ build_features() is ready.")

✅ build_features() is ready.


In [7]:
# ============================================================
# CELL 5: Enter the URL to test
# ============================================================
url = input("🔗 Enter the URL to check: ").strip()
print(f"\nURL received: {url}")

🔗 Enter the URL to check: http://y0utube.cam

URL received: http://y0utube.cam


In [8]:
# ============================================================
# CELL 6: Extract features, load model, and predict
# ============================================================
import pickle

# --- Load the model ---
with open('/content/XGBoostClassifier.pickle.dat', 'rb') as f:
    model = pickle.load(f)

print("🤖 Model loaded successfully.")
print("⏳ Extracting features (WHOIS + HTML fetch may take a few seconds)...\n")

# --- Extract features ---
features = build_features(url)

feature_names = [
    "Have_IP", "Have_At", "URL_Length", "URL_Depth",
    "Redirection", "https_Domain", "TinyURL", "Prefix/Suffix",
    "DNS_Record", "Web_Traffic", "Domain_Age", "Domain_End",
    "iFrame", "Mouse_Over", "Right_Click", "Web_Forwards"
]

print("📊 Extracted Features:")
for name, val in zip(feature_names, features[0]):
    flag = "🔴" if val == 1 else "🟢"
    print(f"  {flag}  {name}: {int(val)}")

# --- Predict ---
prediction = model.predict(features)[0]

print("\n" + "="*45)
if prediction == 0:
    print("✅  Result: This URL is SAFE (Legitimate)")
else:
    print("🚨  Result: This URL is PHISHING — Do NOT visit!")
print("="*45)

🤖 Model loaded successfully.
⏳ Extracting features (WHOIS + HTML fetch may take a few seconds)...

📊 Extracted Features:
  🟢  Have_IP: 0
  🟢  Have_At: 0
  🟢  URL_Length: 0
  🟢  URL_Depth: 0
  🟢  Redirection: 0
  🟢  https_Domain: 0
  🟢  TinyURL: 0
  🟢  Prefix/Suffix: 0
  🔴  DNS_Record: 1
  🔴  Web_Traffic: 1
  🔴  Domain_Age: 1
  🔴  Domain_End: 1
  🟢  iFrame: 0
  🟢  Mouse_Over: 0
  🟢  Right_Click: 0
  🟢  Web_Forwards: 0

🚨  Result: This URL is PHISHING — Do NOT visit!
